In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Data set for DADS June.csv")


def clean_amount(val):
    if isinstance(val, str):
        val = val.replace('₹', '').replace('Rs.', '').replace(',', '').replace('=', '').strip()
    try:
        return float(val)
    except ValueError:
        return np.nan

df['amount_clean'] = df['Amount'].apply(clean_amount)

df['date_clean'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

mask = df['date_clean'].isna()
df.loc[mask, 'date_clean'] = pd.to_datetime(df.loc[mask, 'Date'], errors='coerce')

df['type_clean'] = df['Type'].str.lower().str.strip().replace({'dr': 'debit', 'cr': 'credit'})

df_clean = df.drop_duplicates()

print(f"Cleanup complete. Total valid rows: {len(df_clean)}")
print(f"Total NaT dates remaining: {df_clean['date_clean'].isna().sum()}")

Cleanup complete. Total valid rows: 1310
Total NaT dates remaining: 0


C:\Users\chris\AppData\Local\Temp\ipykernel_38728\3141452983.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.loc[mask, 'date_clean'] = pd.to_datetime(df.loc[mask, 'Date'], errors='coerce')


In [2]:
# Create the vendor mapping dictionary 
vendor_dict = {
    'Swiggy': ['SWIGGY', 'BUNDL'], 
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO', 'KIRANAKART'], 
    'Blinkit': ['BLINKIT', 'GROFERS'], 
    'Amazon': ['AMAZON', 'AMZN'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA', 'FSN'], 
    'Uber': ['UBER'],
    'Ola': ['OLA', 'ANI Technologies'], 
    'Rapido': ['RAPIDO', 'ROPPEN'], 
    'Tummoc': ['TUMMOC'],
    'BMTC': ['BMTC'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'THIRD WAVE', 'TWC'],
    'Cafe Coffee Day': ['CAFE COFFEE DAY', 'CCD', 'COFFEE DAY GLOBAL'],
    'Empire Restaurant': ['EMPIRE'],
    'Truffles': ['TRUFFLES'],
    'Meghana Foods': ['MEGHANA'],
    'Dineout': ['DINEOUT'],
    'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL'], 
    'DMart': ['DMART', 'AVENUE SUPERMARTS'], 
    'Zerodha': ['ZERODHA', 'NEXTBILLION'], 
    'Groww': ['GROWW', 'NEXTBILLION'], 
    'Netflix': ['NETFLIX'],
    'Hotstar': ['HOTSTAR', 'STAR INDIA'],
    'Spotify': ['SPOTIFY'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS', 'BIGTREE'], 
    'Airtel': ['AIRTEL', 'BHARTI AIRTEL'],
    'Jio': ['JIO'],
    'Vodafone Idea': ['VODAFONE', 'VI POSTPAID', 'VI-RECHARGE'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC SUPPLY'],
    'BWSSB': ['BWSSB'],
    'Indian Oil': ['INDIAN OIL', 'IOC'],
    'HP Petrol': ['HP PETROL'],
    'BPCL Petrol': ['BPCL'],
    'P2P Transfer': ['UPI-AMAN', 'UPI-ANKIT', 'UPI-KARAN', 'UPI-NEHA', 'UPI-PRIYA', 'UPI-SNEHA', 'UPI-VIKAS', 'IMPS-RENT-LANDLORD', 'NEFT-TECHCRUSH'],
    'Cash Withdrawal': ['ATM-WDL']
}

def extract_vendor(desc):
    
    if pd.isna(desc):
        return 'Uncategorised'
    
    desc_upper = str(desc).upper()
    
    # Loop through our dictionary
    for canonical_name, keywords in vendor_dict.items():
        for keyword in keywords:
            if keyword.upper() in desc_upper:
                return canonical_name
                
    return 'Uncategorised'

# Apply the function to create a new column
df_clean['vendor_clean'] = df_clean['Description'].apply(extract_vendor)

# Print the top 10 to verify it worked!
print(df_clean['vendor_clean'].value_counts().head(10))

vendor_clean
Swiggy           223
Zomato           121
Ola               87
Amazon            86
Zepto             71
Uber              71
Blinkit           55
Rapido            55
Uncategorised     49
Flipkart          47
Name: count, dtype: int64


C:\Users\chris\AppData\Local\Temp\ipykernel_38728\1100992046.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['vendor_clean'] = df_clean['Description'].apply(extract_vendor)


In [4]:
# ==========================================
# 1. FEATURE 3: Create the 'category' column
# ==========================================
category_dict = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce',
    'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Nykaa': 'E-commerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'Tummoc': 'Transport', 'BMTC': 'Transport',
    'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Cafe Coffee Day': 'Cafe',
    'Empire Restaurant': 'Restaurants', 'Truffles': 'Restaurants', 'Meghana Foods': 'Restaurants', 'Dineout': 'Restaurants',
    'BigBasket': 'Groceries', 'DMart': 'Groceries',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'Netflix': 'Subscriptions', 'Hotstar': 'Subscriptions', 'Spotify': 'Subscriptions',
    'BookMyShow': 'Entertainment',
    'Airtel': 'Utilities', 'Jio': 'Utilities', 'Vodafone Idea': 'Utilities', 'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
    'Indian Oil': 'Fuel', 'HP Petrol': 'Fuel', 'BPCL Petrol': 'Fuel',
    'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal'
}

# Map the vendors to categories (Ensure vendor_clean was successfully created in Feature 2!)
df_clean['category'] = df_clean['vendor_clean'].map(category_dict).fillna('Uncategorised')


# ==========================================
# 2. FEATURE 5: Run the Monthly Trends
# ==========================================
# Extract the month
df_clean['month'] = df_clean['date_clean'].dt.month

# Filter out credits, transfers, and cash withdrawals
spend_df = df_clean[(df_clean['type_clean'] == 'debit') & 
                    (~df_clean['category'].isin(['Personal Transfer', 'Cash Withdrawal']))]

# Build the pivot table
monthly_trends = spend_df.pivot_table(
    values='amount_clean', 
    index='category', 
    columns='month', 
    aggfunc='sum', 
    fill_value=0
)

print("--- MONTHLY TRENDS (₹) ---")
print(monthly_trends.astype(int))

--- MONTHLY TRENDS (₹) ---
month              1      2       3      4       5       6      7      8   \
category                                                                    
Cafe             2682   4233    2953   7151    5430    4862    687   1298   
E-commerce      81124  63093  104923  49023  105854  130678  24411   7855   
Entertainment     311    474     819   1240       0    1017   1514      0   
Food Delivery   22307  21325   21742  23206   21243   26127   3019   1676   
Fuel            30322   2079   24727  18149    7915    2882    705   1437   
Groceries       15787  10837    3964   7733    8830    3166   1257   1862   
Investments     19476   3956   38644  39126   33628    3834  45000  49496   
Quick Commerce   7702  12458   12498   8919   10619    8757    196   1106   
Restaurants     10345   6526   15963  11358   12126   11133   3075   2233   
Subscriptions    2767   4628    1226   2168    1844    2929   1473      0   
Transport       10057   9528    6268   8066   114

C:\Users\chris\AppData\Local\Temp\ipykernel_38728\4126923991.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['category'] = df_clean['vendor_clean'].map(category_dict).fillna('Uncategorised')
C:\Users\chris\AppData\Local\Temp\ipykernel_38728\4126923991.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['month'] = df_clean['date_clean'].dt.month


In [5]:

df_clean['month'] = df_clean['date_clean'].dt.month


spend_df = df_clean[(df_clean['type_clean'] == 'debit') & 
                    (~df_clean['category'].isin(['Personal Transfer', 'Cash Withdrawal']))]


monthly_trends = spend_df.pivot_table(
    values='amount_clean', 
    index='category', 
    columns='month', 
    aggfunc='sum', 
    fill_value=0
)

print("--- MONTHLY TRENDS (₹) ---")
print(monthly_trends.astype(int)) # Converted to int for cleaner reading

--- MONTHLY TRENDS (₹) ---
month              1      2       3      4       5       6      7      8   \
category                                                                    
Cafe             2682   4233    2953   7151    5430    4862    687   1298   
E-commerce      81124  63093  104923  49023  105854  130678  24411   7855   
Entertainment     311    474     819   1240       0    1017   1514      0   
Food Delivery   22307  21325   21742  23206   21243   26127   3019   1676   
Fuel            30322   2079   24727  18149    7915    2882    705   1437   
Groceries       15787  10837    3964   7733    8830    3166   1257   1862   
Investments     19476   3956   38644  39126   33628    3834  45000  49496   
Quick Commerce   7702  12458   12498   8919   10619    8757    196   1106   
Restaurants     10345   6526   15963  11358   12126   11133   3075   2233   
Subscriptions    2767   4628    1226   2168    1844    2929   1473      0   
Transport       10057   9528    6268   8066   114

C:\Users\chris\AppData\Local\Temp\ipykernel_38728\4028962880.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['month'] = df_clean['date_clean'].dt.month


In [6]:
import numpy as np

# 1. Filter to debits only for our anomaly hunt
debits_df = df_clean[df_clean['type_clean'] == 'debit'].copy()

# 2. Calculate the mean and standard deviation for EACH category
debits_df['category_mean'] = debits_df.groupby('category')['amount_clean'].transform('mean')
debits_df['category_std'] = debits_df.groupby('category')['amount_clean'].transform('std')

# 3. Calculate the Z-score (handling potential division by zero for categories with only 1 item)
debits_df['z_score'] = np.where(
    debits_df['category_std'] > 0,
    (debits_df['amount_clean'] - debits_df['category_mean']) / debits_df['category_std'],
    0
)

# 4. Filter for anomalies (Z-score > 2) and sort them
anomalies = debits_df[debits_df['z_score'] > 2].sort_values(by='z_score', ascending=False)

print("--- TOP 5 ANOMALOUS TRANSACTIONS ---")
print(anomalies[['date_clean', 'vendor_clean', 'category', 'amount_clean', 'z_score']].head())

--- TOP 5 ANOMALOUS TRANSACTIONS ---
     date_clean   vendor_clean       category  amount_clean   z_score
414  2024-02-26  Uncategorised  Uncategorised        8383.0  4.915571
49   2024-07-01  Uncategorised  Uncategorised        7314.0  4.194253
1298 2024-06-26         Amazon     E-commerce       22008.0  4.090349
269  2024-07-02         Amazon     E-commerce       21986.0  4.085484
1271 2024-06-22        Dineout    Restaurants        7935.0  3.637947


In [8]:
# ==========================================
# FEATURE 8: Spending Archetype Detection
# ==========================================
import numpy as np

# 1. Re-calculate core variables (Safeguard against kernel restarts)
credits = df_clean[df_clean['type_clean'] == 'credit']['amount_clean'].sum()
debits = df_clean[df_clean['type_clean'] == 'debit']['amount_clean'].sum()
net_change = credits - debits
savings_rate = (net_change / credits) * 100 if credits > 0 else 0

# 2. Filter to debits only for our archetype hunt
debits_df = df_clean[df_clean['type_clean'] == 'debit'].copy()

# 3. Calculate specific category spends
food_spend = debits_df[debits_df['category'].isin(['Food Delivery', 'Restaurants', 'Cafe'])]['amount_clean'].sum()
qcom_spend = debits_df[debits_df['category'] == 'Quick Commerce']['amount_clean'].sum()
ecom_spend = debits_df[debits_df['category'] == 'E-commerce']['amount_clean'].sum()
inv_spend = debits_df[debits_df['category'] == 'Investments']['amount_clean'].sum()
transport_spend = debits_df[debits_df['category'] == 'Transport']['amount_clean'].sum()

# Count unique subscription vendors
active_subs = debits_df[debits_df['category'] == 'Subscriptions']['vendor_clean'].nunique()

# Re-calculate Late Night Percentage (from Feature 6)
debits_df['hour'] = debits_df['Time'].astype(str).str[:2].astype(int)
food_df = debits_df[debits_df['category'] == 'Food Delivery']
late_night_food = food_df[(food_df['hour'] >= 21) | (food_df['hour'] <= 2)]
late_night_pct = (len(late_night_food) / len(food_df)) * 100 if len(food_df) > 0 else 0

# 4. Archetype Logic Rules Checklist
archetypes = []

# THE FOODIE: Food + Rest + Cafe > 25% of debits
if (food_spend / debits) > 0.25:
    archetypes.append(f"THE FOODIE ({(food_spend/debits)*100:.1f}% on food)")

# THE QUICK COMMERCE JUNKIE: Q-com > 15% of debits
if (qcom_spend / debits) > 0.15:
    archetypes.append(f"THE QUICK COMMERCE JUNKIE ({(qcom_spend/debits)*100:.1f}% on Q-com)")

# THE SHOPAHOLIC: E-commerce > 15% of debits
if (ecom_spend / debits) > 0.15:
    archetypes.append(f"THE SHOPAHOLIC ({(ecom_spend/debits)*100:.1f}% on e-commerce)")

# THE INVESTOR: Investments > 15% of debits
if (inv_spend / debits) > 0.15:
    archetypes.append(f"THE INVESTOR ({(inv_spend/debits)*100:.1f}% on SIPs/Investments)")

# THE LATE-NIGHT SNACKER: >50% food delivery is 9PM-2AM
if late_night_pct > 50:
    archetypes.append(f"THE LATE-NIGHT SNACKER ({late_night_pct:.1f}% food ordered late)")

# THE CAB COMMUTER: Transport > 10% of debits
if (transport_spend / debits) > 0.10:
    archetypes.append("THE CAB COMMUTER")

# THE SUBSCRIPTION LOVER: 5 or more active subs
if active_subs >= 5:
    archetypes.append(f"THE SUBSCRIPTION LOVER ({active_subs} active subs)")

# THE YOLO SPENDER vs DISCIPLINED SAVER
if savings_rate < 10:
    archetypes.append(f"THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")
elif savings_rate > 40:
    archetypes.append("THE DISCIPLINED SAVER")

print("--- RAHUL'S SPENDING ARCHETYPES ---")
for arc in archetypes:
    print(f"-> {arc}")

--- RAHUL'S SPENDING ARCHETYPES ---
-> THE SHOPAHOLIC (36.0% on e-commerce)
-> THE YOLO SPENDER (savings rate -229.3%)
